In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=b7f0f5dfaad35ba3a528a5ffd078e27f4e01b4a60ae4cb70a2a30738e0cb0938
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import h2o
from h2o.automl import H2OAutoML

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn import tree
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
from lifelines import KaplanMeierFitter

def calculate_survival_probabilities(df, time_col, event_col):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df['target'] = calculate_survival_probabilities(df, time_col, event_col)
    df.loc[df[event_col] == 0, 'target'] -= 0.2  # Adjust for censored data
    return df

train = preprocess_survival_data(train)

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        if i[0]!='efs' or i[0]!='efs_time':
            train[i[0]] = train[i[0]].fillna(0)
            #train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))
    else:
        if i[0]!='efs' or i[0]!='efs_time':
            test[i[0]] = test[i[0]].fillna(0)
            #test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.24" 2024-07-16; OpenJDK Runtime Environment (build 11.0.24+8-post-Ubuntu-1ubuntu322.04); OpenJDK 64-Bit Server VM (build 11.0.24+8-post-Ubuntu-1ubuntu322.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.10/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp5w_2wuij
  JVM stdout: /tmp/tmp5w_2wuij/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp5w_2wuij/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,2 months and 2 days
H2O_cluster_name:,H2O_from_python_unknownUser_l27rqw
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.500 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


In [9]:
train_x = train.drop(columns=['efs', 'efs_time', 'target'])
train_x = train_x[['dri_score','psych_disturb','cyto_score','diabetes','tbi_status','graft_type','prim_disease_hct','cmv_status','prod_type',
'cyto_score_detail','conditioning_intensity','year_hct','in_vivo_tcd','donor_age','age_at_hct','gvhd_proph','sex_match',
'comorbidity_score','karnofsky_score']]
train_y = train['target']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.001, random_state=100)

for i in train_x:
    model=XGBRegressor()
    print(i)
    model.fit(X_train[i].to_numpy().reshape(-1, 1),y_train)
    print(mean_squared_error(y_test,model.predict(X_test[i].to_numpy().reshape(-1, 1))))

    plt.plot(y_test.to_numpy().reshape(-1))
    plt.plot(model.predict(X_test[i]))
    plt.show()

In [11]:
['dri_score','psych_disturb','cyto_score','diabetes','tbi_status','graft_type','prim_disease_hct','cmv_status','prod_type',
'cyto_score_detail','conditioning_intensity','year_hct','in_vivo_tcd','donor_age','age_at_hct','gvhd_proph','sex_match',
'comorbidity_score','karnofsky_score']

['dri_score',
 'psych_disturb',
 'cyto_score',
 'diabetes',
 'tbi_status',
 'graft_type',
 'prim_disease_hct',
 'cmv_status',
 'prod_type',
 'cyto_score_detail',
 'conditioning_intensity',
 'year_hct',
 'in_vivo_tcd',
 'donor_age',
 'age_at_hct',
 'gvhd_proph',
 'sex_match',
 'comorbidity_score',
 'karnofsky_score']

In [12]:
x,y=list(train_x.columns),'target'

In [13]:
catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)



cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)
        
catboost.fit(train_pool, eval_set=(val_pool), verbose=50, early_stopping_rounds=100)

Learning rate set to 0.086288
0:	learn: 0.2578254	test: 0.2300532	best: 0.2300532 (0)	total: 70.2ms	remaining: 1m 10s
50:	learn: 0.2326552	test: 0.2165110	best: 0.2134039 (17)	total: 768ms	remaining: 14.3s
100:	learn: 0.2269800	test: 0.2151858	best: 0.2134039 (17)	total: 1.37s	remaining: 12.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2134039326
bestIteration = 17

Shrink model to first 18 iterations.


In [14]:
cat_pre_test=catboost.predict(test)
#print(f'Concordance index (lifelines): {ci_lifelines(y_test, catboost.predict(X_test))}')

In [15]:
#Concordance index (lifelines): 0.6434623996950867

In [16]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = (cat_pre_test.reshape(-1))
test_predictions

array([0.47402558, 0.60985118, 0.47365217])

In [17]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.474026
1,28801,0.609851
2,28802,0.473652


In [18]:
submission.to_csv('submission.csv', index = False)